# PyTorch: Classification

We'll use the same circular decision boundary dataset from our forward propagation demo, but now train the network with PyTorch and evaluate on a test set.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
except ImportError:
    %pip install torch
    import torch

import torch.nn as nn

In [ ]:
# Generate random points in a 2D space
np.random.seed(20)

x1 = np.random.uniform(-1.5, 1.5, 200)
x2 = np.random.uniform(-1.5, 1.5, 200)

# Label: 1 if outside the unit circle, 0 if inside
y = (x1**2 + x2**2 >= 1).astype(int)

dataset = np.column_stack((x1, x2, y))

In [ ]:
def make_scatter_plot_with_shading(X_plot, y_plot, model=None):
    plt.figure(figsize=(6, 6))
    for label, marker, color in zip([0, 1], ['o', 'x'], ['blue', 'red']):
        mask = y_plot == label
        plt.scatter(X_plot[mask, 0], X_plot[mask, 1], marker=marker, color=color, label='Class ' + str(label), s=100)

    if model is not None:
        x1space = np.linspace(-1.5, 1.5, 100)
        x2space = np.linspace(-1.5, 1.5, 100)
        X1, X2 = np.meshgrid(x1space, x2space)
        grid = torch.tensor(np.column_stack((X1.ravel(), X2.ravel())), dtype=torch.float32)
        with torch.no_grad():
            Z = model(grid).numpy().reshape(X1.shape)
        plt.contourf(X1, X2, Z, levels=[0, 0.5, 1], colors=['blue', 'red'], alpha=0.3)

    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.legend()
    plt.show()

make_scatter_plot_with_shading(dataset[:, :2], dataset[:, 2])

# Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X = dataset[:, :2]
y = dataset[:, 2]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

# to use PyTorch's binary cross-entropy loss: y must be float32, shape (n, 1)
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)  
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

print("Training set:", X_train.shape[0], "points")
print("Test set:", X_test.shape[0], "points")

# Define and train the model

In [ ]:
# torch.manual_seed(1)

model = nn.Sequential(
    nn.Linear(2, 5),
    nn.ReLU(),
    nn.Linear(5, 1),
    nn.Sigmoid()          # squashes output to [0, 1] for classification
)

# BCELoss = binary cross-entropy, the classification equivalent of MSELoss
loss_fn = nn.BCELoss()

# Despite the name, this is batch gradient descent since we pass all data each iteration
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

print(model)

losses = []
for epoch in range(3000):
    y_hat = model(X_train)
    loss = loss_fn(y_hat, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

print("Final training loss:", losses[-1])

In [ ]:
plt.scatter(range(0, len(losses)), losses)
plt.show()

# Evaluate on train and test sets

In [ ]:
model.eval()      # set model to evaluation mode

def compute_accuracy(X_data, y_data):
    with torch.no_grad():
        y_hat = model(X_data)

        # converts each probability into just True/False, then turn into floats (0.0 or 1.0)
        predictions = (y_hat >= 0.5).float()  # tensor of same shape as original y_hat, just 0.0 or 1.0 for each value 

        # tests each prediction vs true y value, each one becomes True/False, then turn into floats and take average
        accuracy = (predictions == y_data).float().mean()
    return accuracy.item()  # accuracy is still a tensor, so .item() grabs the single item in it

print("Train accuracy:", compute_accuracy(X_train, y_train))
print("Test accuracy: ", compute_accuracy(X_test, y_test))

In [ ]:
# Decision boundary on training data
make_scatter_plot_with_shading(X_train.numpy(), y_train.numpy().ravel(), model=model)

In [ ]:
# Decision boundary on test data
make_scatter_plot_with_shading(X_test.numpy(), y_test.numpy().ravel(), model=model)

# Multi-class classification (3 classes)

Same idea, but now with 3 concentric regions:
- Class 0: inside radius 0.7
- Class 1: between radius 0.7 and 1.2
- Class 2: outside radius 1.2

Key differences from binary classification:
- Output layer has **3 neurons** (one per class) instead of 1
- No `Sigmoid` — use `nn.CrossEntropyLoss()` which applies softmax internally
- Labels are **integers** (0, 1, 2), not floats

In [ ]:
# Generate 3-class dataset
# np.random.seed(1)

x1 = np.random.uniform(-1.5, 1.5, 300)
x2 = np.random.uniform(-1.5, 1.5, 300)

radius = np.sqrt(x1**2 + x2**2)
y = np.where(radius < 0.7, 0, np.where(radius < 1.2, 1, 2))

dataset3 = np.column_stack((x1, x2, y))

In [ ]:
def make_scatter_plot_3class(X_plot, y_plot, model=None):
    plt.figure(figsize=(6, 6))
    colors = ['blue', 'green', 'red']
    markers = ['o', 's', 'x']
    for label in [0, 1, 2]:
        mask = y_plot == label
        plt.scatter(X_plot[mask, 0], X_plot[mask, 1], marker=markers[label],
                    color=colors[label], label='Class ' + str(label), s=100)

    if model is not None:
        x1space = np.linspace(-1.5, 1.5, 500)
        x2space = np.linspace(-1.5, 1.5, 500)
        X1, X2 = np.meshgrid(x1space, x2space)
        grid = torch.tensor(np.column_stack((X1.ravel(), X2.ravel())), dtype=torch.float32)
        with torch.no_grad():
            logits = model(grid)
            Z = logits.argmax(dim=1).numpy().reshape(X1.shape)
        plt.contourf(X1, X2, Z, levels=[-0.5, 0.5, 1.5, 2.5],
                     colors=colors, alpha=0.2)

    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.legend()
    plt.show()

make_scatter_plot_3class(dataset3[:, :2], dataset3[:, 2])

In [ ]:
X = dataset3[:, :2]
y = dataset3[:, 2]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

# CrossEntropyLoss expects integer labels (torch.long), not floats, and only 1-dimensional
# (very frustrating, because 2-class binary cross entropy wanted floats in 2d)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

print("Training set:", X_train.shape[0], "points")
print("Test set:", X_test.shape[0], "points")

In [ ]:
# torch.manual_seed(1)

model = nn.Sequential(
    nn.Linear(2, 5),
    nn.ReLU(),
    nn.Linear(5, 3)      # 3 outputs, one per class 
)

# Notice we don't add softmax as a layer, unlike when we had 2 classes above.
# CrossEntropyLoss applies softmax internally, so the model outputs raw scores (known as "logits")
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

print(model)

losses = []
for epoch in range(3000):
    y_hat = model(X_train)         # output shape: (n, 3) — one score per class
    loss = loss_fn(y_hat, y_train) # CrossEntropyLoss compares raw output to integer labels

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

print("Final training loss:", losses[-1])

In [ ]:
plt.scatter(range(0, len(losses)), losses)
plt.show()

In [ ]:
model.eval()

def compute_accuracy(X_data, y_data):
    with torch.no_grad():
        y_hat = model(X_data)
        predictions = y_hat.argmax(dim=1)  # pick the class with the highest score
        accuracy = (predictions == y_data).float().mean()
    return accuracy.item()

print("Train accuracy:", compute_accuracy(X_train, y_train))
print("Test accuracy: ", compute_accuracy(X_test, y_test))

In [ ]:
# Decision boundary on training data
make_scatter_plot_3class(X_train.numpy(), y_train.numpy(), model=model)

In [ ]:
# Decision boundary on test data
make_scatter_plot_3class(X_test.numpy(), y_test.numpy(), model=model)